In [ ]:
using Sunny, CairoMakie, ColorSchemes, Colors, LinearAlgebra, ProgressMeter
# Cairo for static, GL for interactive

include("/Users/fbnielsen/Library/CloudStorage/OneDrive-UniversityofCopenhagen/Uni OneDrive/4 år/Clinoatacamite/Sunny_models/func_clino1.jl")
units = Units(:meV, :angstrom)

# Final Q-Mag project

## Spin Ice lattice Holmiumtitenate  $(\text{Ho}_2\text{Ti}_2\text{O}_7)$

### Definition of crystal

In [ ]:
latvecs = lattice_vectors(7.175, 7.175, 7.175, 90, 90, 90) #https://legacy.materialsproject.org/materials/mp-33948/

positions = [[0.875, 0.625, 0.375],
             [0.625, 0.125, 0.625],
             [0.875, 0.875, 0.125],
             [0.625, 0.875, 0.375],
             [0.875, 0.125, 0.875],
             [0.625, 0.625, 0.125],
             [0.875, 0.375, 0.625],
             [0.625, 0.375, 0.875],
             [0.375, 0.625, 0.875],
             [0.125, 0.125, 0.125],
             [0.375, 0.875, 0.625],
             [0.125, 0.875, 0.875],
             [0.375, 0.125, 0.375],
             [0.125, 0.625, 0.625],
             [0.375, 0.375, 0.125],
             [0.125, 0.375, 0.375]
             ] # From other pyrochlore material: https://sunnysuite.github.io/Sunny.jl/stable/examples/contributed/MgCr2O4-tutorial.html#Setting-up-the-crystal-structure

types = ["Ho" for _ in positions]

cryst = Crystal(latvecs, positions; types=types)  # Fd3m [227]

sys = System(cryst, [1 => Moment(s=2, g=5/4)] , :dipole) 

sys = reshape_supercell(sys, primitive_cell(cryst))
sys = reshape_supercell(sys, [1 0 0; 0 1 0; 0 0 1])

UndefVarError: UndefVarError: `lattice_vectors` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
view_crystal(sys)

UndefVarError: UndefVarError: `view_crystal` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
J_nn = 0.52 / 11.604
set_exchange!(sys, J_nn, Bond(1, 3, [0,0,0]))

D = 2.35 /11.604
set_onsite_coupling!(sys, -D * S[3]^2, i) 

sys_lr = clone_system(sys)
enable_dipole_dipole!(sys_lr, units.vacuum_permeability)

sys_extend = reshape_supercell(sys_lr, [10 0 0; 0 10 0; 0 0 10])
randomize_spins!(sys_extend)
minimize_energy!(sys_extend, maxiters=10000)

display(energy_per_site(sys_extend))

### Langevin

In [ ]:
kT      = 1.7 / 11            
tol     = 1e-3                 
nsteps  = 1000

dt      = 1e-4    
damping = 0.1     
kT      = 1.7/11


sys_i = reshape_supercell(sys_lr, [10 0 0; 0 10 0; 0 0 10]) 
randomize_spins!(sys_i)

fig2 = Figure(size=(1000, 500))

langevin = Langevin(dt; damping, kT)
suggest_timestep(sys_i, langevin; tol=tol)

energies = [energy_per_site(sys_i)]
for _ in 1:nsteps
    step!(sys_i, langevin)
    push!(energies, energy_per_site(sys_i))
end
ax = Axis(fig2[1,1];
    xlabel = "Timesteps",
    ylabel = "Energy (meV)",
    title  = "dt = $(dt), damping = $(damping), tol = $(tol)")
lines!(ax, energies, color=:red)


fig2


UndefVarError: UndefVarError: `reshape_supercell` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
#Sample
kT      = 1.7*Sunny.meV_per_K
dt      = 1e-4
damping = 0.1
si=3 # q-space pixel size as 1/si

langevin = Langevin(dt; damping, kT)

# Energy range: max resolvable frequency ~ π/dt, but stay physically reasonable

energies = range(0, 5, 100)

sys_i = reshape_supercell(sys_lr, [si 0 0; 0 si 0; 0 0 si]) 
randomize_spins!(sys_i)

formfactors = [1 => FormFactor("Ho3")]
sc = SampledCorrelations(sys_i; dt, energies, measure=ssf_perp(sys_i; formfactors))

n_decorr  = 600   # steps between samples (matching your thermalization)
n_samples = 35     # more samples = smoother spectrum

@showprogress for sample in 1:n_samples #for sample in 1:n_samples
    for _ in 1:n_decorr
        step!(sys_i, langevin)
    end
    add_sample!(sc, sys_i)
end

UndefVarError: UndefVarError: `Sunny` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
#Plot (Q,E)-colormap (energies are chosen, q's aren't yet)
fig = Figure(size=(800, 600))
qbs = 0.0


qbs  = 0.0:0.5:1.5
dirs = ["h", "h", "h", "h"]  # adjust per slice
npts = 200

fig = Figure(size=(1200, 900))

for (i, qb) in enumerate(qbs)
    path = [[-4, qb, 0], [4, qb, 0]]
    qpts = q_space_path(cryst, path, npts)
    Sqw  = intensities(sc, qpts; energies=:available, kT)

    xticks, xlabels = qticks_lang(5, path, npts, dirs[i])
    emax = energies[end]

    plot_intensities!(fig[fldmod1(i, 2)...], Sqw;
        title = "K = $qb",
        colormap = :viridis,
        axis = (
            yticks      = 0:emax/emax:emax,
            xticks      = (xticks, xlabels),
            xlabel      = "(H $(qb) 0) (r.l.u.)",
            ylabel      = "Energy (meV)",
        )
    )
end

Label(fig[0, 1:2], "Jnn = $(J_nn), Ising-SIA = $(D), with dipole-dipole \nSystem size: $(si)x$(si)x$(si), tol: $(tol), damping: $(damping), dt: $(dt), T = $(kT/Sunny.meV_per_K) K"; fontsize=15, font=:bold)

fig

UndefVarError: UndefVarError: `Figure` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
#Plot integrated over all E (Q,Q)-cut (Static intensities)
qpts = q_space_grid(cryst, [1, 1, 0], range(-6, 6, 200), [0, 0, 1], (-6, 6))
Sq  = intensities_static(sc, qpts; kT)

fig = Figure(; size=(700,500))
plot_intensities!(fig[1,1], Sq; title="Jnn = $(J_nn), Ising-SIA = $(D), with dipole-dipole \nStatic/integrated 0<E<$(energies[end]) meV \nSystem size: $(si)x$(si)x$(si), tol: $(tol), damping: $(damping), dt: $(dt), T = $(round(kT/Sunny.meV_per_K; digits=3)) K", colormap=:viridis)
ax = current_axis()
ax.aspect = DataAspect()
fig

UndefVarError: UndefVarError: `q_space_grid` not defined in `Main`
Suggestion: check for spelling errors or missing imports.